# MeanRev VWAP Supertrend Strategy

Port of the mean-reversion VWAP band and Supertrend flip signal logic.

Source strategy: `/home/amit/python/src/github.com/AppsByZubin/garageforbots/index/strategy/meanrev_vwap_supertrend.py`

Candles are loaded from `data/processed/nifty50_2026-01-01_2026-04-30.csv`. The matching Nifty futures CSV is used only for the volume fields required by the original strategy formulas.

In [ ]:
from pathlib import Path
from datetime import time
import os

os.environ.setdefault("PANDAS_USE_NUMEXPR", "0")
os.environ.setdefault("PANDAS_USE_BOTTLENECK", "0")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.options.display.float_format = "{:,.4f}".format


## Load Data

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate the visualizer project root.")


PROJECT_ROOT = find_project_root()
SPOT_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "nifty50_2026-01-01_2026-04-30.csv"
FUTURE_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "nifty50_future_2026-01-01_2026-04-30.csv"


def load_market_data() -> pd.DataFrame:
    if not SPOT_CSV_PATH.exists():
        raise FileNotFoundError(f"Missing spot data CSV: {SPOT_CSV_PATH}")

    spot = pd.read_csv(SPOT_CSV_PATH, parse_dates=["timestamp"])
    required = {"timestamp", "open", "high", "low", "close", "volume", "open_interest"}
    missing = required.difference(spot.columns)
    if missing:
        raise ValueError(f"Spot CSV is missing required column(s): {sorted(missing)}")

    spot = spot.rename(
        columns={
            "volume": "nifty_volume",
            "open_interest": "nifty_open_interest",
        }
    )

    price_columns = ["open", "high", "low", "close"]
    spot[price_columns] = spot[price_columns].apply(pd.to_numeric, errors="coerce")
    spot["nifty_volume"] = pd.to_numeric(spot["nifty_volume"], errors="coerce").fillna(0.0)
    spot["nifty_open_interest"] = pd.to_numeric(spot["nifty_open_interest"], errors="coerce").fillna(0.0)
    spot = spot.dropna(subset=["timestamp", *price_columns])
    spot = spot.sort_values("timestamp").drop_duplicates("timestamp", keep="last")

    if FUTURE_CSV_PATH.exists():
        future = pd.read_csv(FUTURE_CSV_PATH, parse_dates=["timestamp"])
        future = future.rename(
            columns={
                "volume": "future_volume",
                "open_interest": "future_open_interest",
            }
        )
        future["future_volume"] = pd.to_numeric(future["future_volume"], errors="coerce").fillna(0.0)
        future["future_open_interest"] = pd.to_numeric(
            future["future_open_interest"], errors="coerce"
        ).fillna(0.0)
        future = future[["timestamp", "future_volume", "future_open_interest"]]
        future = future.sort_values("timestamp").drop_duplicates("timestamp", keep="last")
        data = spot.merge(future, on="timestamp", how="left", validate="one_to_one")
    else:
        data = spot.copy()
        data["future_volume"] = np.nan
        data["future_open_interest"] = np.nan

    data["fut_volume"] = pd.to_numeric(data["future_volume"], errors="coerce")
    data["spot_volume"] = pd.to_numeric(data["nifty_volume"], errors="coerce")
    data["effective_volume"] = data["fut_volume"].where(data["fut_volume"] > 0, data["spot_volume"])
    data["effective_volume"] = data["effective_volume"].where(data["effective_volume"] > 0, 1.0)
    data["session_date"] = data["timestamp"].dt.date
    data["session_time"] = data["timestamp"].dt.strftime("%H:%M")
    return data.reset_index(drop=True)


market_data = load_market_data()
print(f"Loaded spot candles from {SPOT_CSV_PATH.relative_to(PROJECT_ROOT)}")
if FUTURE_CSV_PATH.exists():
    print(f"Using matching futures volume from {FUTURE_CSV_PATH.relative_to(PROJECT_ROOT)}")
else:
    print("Matching futures volume CSV not found; using equal-volume fallback for volume-weighted indicators.")
print(f"{len(market_data):,} rows from {market_data['timestamp'].min()} to {market_data['timestamp'].max()}")
display(market_data.head())


## Indicator Helpers

In [ ]:
def wilder_rma(series: pd.Series, length: int) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce").astype(float)
    return numeric.ewm(alpha=1 / float(length), adjust=False, min_periods=length).mean()


def ema(series: pd.Series, length: int) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce").astype(float)
    return numeric.ewm(span=int(length), adjust=False, min_periods=int(length)).mean()


def wma(series: pd.Series, length: int) -> pd.Series:
    length = int(length)
    numeric = pd.to_numeric(series, errors="coerce").astype(float)
    weights = np.arange(1, length + 1, dtype="float64")
    return numeric.rolling(window=length, min_periods=length).apply(
        lambda values: float(np.dot(values, weights) / weights.sum()),
        raw=True,
    )


def rsi(close: pd.Series, length: int = 7) -> pd.Series:
    close = pd.to_numeric(close, errors="coerce").astype(float)
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = (-delta).clip(lower=0.0)
    avg_gain = wilder_rma(gain, length)
    avg_loss = wilder_rma(loss, length)
    rs = avg_gain / avg_loss.replace(0, np.nan)
    out = 100.0 - (100.0 / (1.0 + rs))
    out = out.mask((avg_loss == 0) & (avg_gain > 0), 100.0)
    out = out.mask((avg_gain == 0) & (avg_loss > 0), 0.0)
    out = out.mask((avg_gain == 0) & (avg_loss == 0), 50.0)
    return out.replace([np.inf, -np.inf], np.nan)


def atr(high: pd.Series, low: pd.Series, close: pd.Series, length: int = 14) -> pd.Series:
    high = pd.to_numeric(high, errors="coerce").astype(float)
    low = pd.to_numeric(low, errors="coerce").astype(float)
    close = pd.to_numeric(close, errors="coerce").astype(float)
    prev_close = close.shift(1)
    true_range = pd.concat(
        [high - low, (high - prev_close).abs(), (low - prev_close).abs()],
        axis=1,
    ).max(axis=1)
    return wilder_rma(true_range, length)


def adx(high: pd.Series, low: pd.Series, close: pd.Series, length: int = 14) -> pd.Series:
    high = pd.to_numeric(high, errors="coerce").astype(float)
    low = pd.to_numeric(low, errors="coerce").astype(float)
    close = pd.to_numeric(close, errors="coerce").astype(float)

    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = pd.Series(np.where((up_move > down_move) & (up_move > 0), up_move, 0.0), index=high.index)
    minus_dm = pd.Series(np.where((down_move > up_move) & (down_move > 0), down_move, 0.0), index=high.index)
    tr = pd.concat(
        [high - low, (high - close.shift(1)).abs(), (low - close.shift(1)).abs()],
        axis=1,
    ).max(axis=1)
    smoothed_tr = wilder_rma(tr, length).replace(0, np.nan)
    plus_di = 100.0 * wilder_rma(plus_dm, length) / smoothed_tr
    minus_di = 100.0 * wilder_rma(minus_dm, length) / smoothed_tr
    dx = 100.0 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    return wilder_rma(dx, length)


def angle(series: pd.Series, window: int = 3) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce").astype(float)
    slope = (numeric - numeric.shift(window)) / float(window)
    return pd.Series(np.degrees(np.arctan(np.clip(slope, -10, 10))), index=series.index)


def calculate_supertrend(
    high: pd.Series,
    low: pd.Series,
    close: pd.Series,
    atr_period: int = 10,
    factor: float = 3.0,
) -> pd.DataFrame:
    high_s = pd.to_numeric(high, errors="coerce").astype("float64").reset_index(drop=True)
    low_s = pd.to_numeric(low, errors="coerce").astype("float64").reset_index(drop=True)
    close_s = pd.to_numeric(close, errors="coerce").astype("float64").reset_index(drop=True)
    atr_s = atr(high_s, low_s, close_s, length=int(atr_period)).reset_index(drop=True)

    hl2 = (high_s + low_s) / 2.0
    basic_upper = hl2 + (float(factor) * atr_s)
    basic_lower = hl2 - (float(factor) * atr_s)
    final_upper = pd.Series(np.nan, index=high_s.index, dtype="float64")
    final_lower = pd.Series(np.nan, index=high_s.index, dtype="float64")
    direction = pd.Series(np.nan, index=high_s.index, dtype="float64")
    supertrend = pd.Series(np.nan, index=high_s.index, dtype="float64")

    first_valid = atr_s.first_valid_index()
    if first_valid is None:
        return pd.DataFrame(
            {
                "st_atr_10": atr_s,
                "st_upperbound": final_upper,
                "st_lowerbound": final_lower,
                "supertrend": supertrend,
                "st_direction": direction,
            }
        )

    for i in range(int(first_valid), len(high_s)):
        if pd.isna(basic_upper.iloc[i]) or pd.isna(basic_lower.iloc[i]) or pd.isna(close_s.iloc[i]):
            continue

        if i == int(first_valid):
            final_upper.iloc[i] = basic_upper.iloc[i]
            final_lower.iloc[i] = basic_lower.iloc[i]
            direction.iloc[i] = 1.0
        else:
            prev_upper = final_upper.iloc[i - 1]
            prev_lower = final_lower.iloc[i - 1]
            prev_close = close_s.iloc[i - 1]
            prev_supertrend = supertrend.iloc[i - 1]

            if pd.isna(prev_upper):
                final_upper.iloc[i] = basic_upper.iloc[i]
            elif basic_upper.iloc[i] < prev_upper or prev_close > prev_upper:
                final_upper.iloc[i] = basic_upper.iloc[i]
            else:
                final_upper.iloc[i] = prev_upper

            if pd.isna(prev_lower):
                final_lower.iloc[i] = basic_lower.iloc[i]
            elif basic_lower.iloc[i] > prev_lower or prev_close < prev_lower:
                final_lower.iloc[i] = basic_lower.iloc[i]
            else:
                final_lower.iloc[i] = prev_lower

            if pd.isna(atr_s.iloc[i - 1]):
                direction.iloc[i] = 1.0
            elif prev_supertrend == prev_upper:
                direction.iloc[i] = -1.0 if close_s.iloc[i] > final_upper.iloc[i] else 1.0
            else:
                direction.iloc[i] = 1.0 if close_s.iloc[i] < final_lower.iloc[i] else -1.0

        supertrend.iloc[i] = final_lower.iloc[i] if direction.iloc[i] < 0 else final_upper.iloc[i]

    if len(supertrend) > 0:
        supertrend.iloc[0] = np.nan

    return pd.DataFrame(
        {
            "st_atr_10": atr_s,
            "st_upperbound": final_upper,
            "st_lowerbound": final_lower,
            "supertrend": supertrend,
            "st_direction": direction,
        }
    )


def add_price_action(df: pd.DataFrame, volatile_range: float = 8.0, major_move_default: float = 10.0) -> pd.DataFrame:
    df = df.copy()
    df["candle_range"] = df["high"] - df["low"]
    df["volatile_count"] = (df["candle_range"] > volatile_range).astype(int).shift(1).rolling(window=4).sum()
    df["is_volatile"] = df["volatile_count"] >= 2
    df["recent_high_max"] = df["high"].shift(1).rolling(window=4).max()
    df["recent_low_min"] = df["low"].shift(1).rolling(window=4).min()
    df["is_hh"] = (df["high"] > df["recent_high_max"]) & df["is_volatile"]
    df["is_ll"] = (df["low"] < df["recent_low_min"]) & df["is_volatile"]

    is_red = df["close"] < df["open"]
    is_green = df["close"] > df["open"]
    curr_range = df["candle_range"]
    prev_range = curr_range.shift(1)
    is_alive = (curr_range > 3) & (prev_range > 3)
    threshold = pd.to_numeric(df.get("atr_14", major_move_default), errors="coerce")
    if not isinstance(threshold, pd.Series):
        threshold = pd.Series(major_move_default, index=df.index)
    threshold = threshold.fillna(major_move_default)
    has_major_move = (curr_range > threshold) | (prev_range > threshold)
    is_valid_setup = is_alive & has_major_move

    df["is_bearish_thrust"] = is_red & is_red.shift(1) & (df["low"] < df["low"].shift(1)) & is_valid_setup
    df["is_bullish_thrust"] = is_green & is_green.shift(1) & (df["high"] > df["high"].shift(1)) & is_valid_setup
    return df


def signal_events(raw_signal: pd.Series) -> pd.Series:
    raw_signal = raw_signal.fillna("")
    return raw_signal.where((raw_signal != "") & (raw_signal != raw_signal.shift()), "")


In [ ]:
def first_signal_or_latest_day(strategy: pd.DataFrame) -> str:
    signal_days = strategy.loc[strategy["signal"].ne(""), "session_date"].astype(str).unique()
    if len(signal_days):
        return str(signal_days[0])
    return str(strategy["session_date"].iloc[-1])


def display_signal_summary(strategy: pd.DataFrame, columns: list[str]) -> None:
    events = strategy.loc[strategy["signal"].ne(""), columns].copy()
    print(f"{len(events):,} signal event(s)")
    if events.empty:
        return

    daily_counts = pd.crosstab(
        strategy.loc[strategy["signal"].ne(""), "session_date"],
        strategy.loc[strategy["signal"].ne(""), "signal"],
    )
    for side in ["CALL", "PUT"]:
        if side not in daily_counts.columns:
            daily_counts[side] = 0
    daily_counts = daily_counts[["CALL", "PUT"]].sort_index()
    display(daily_counts.tail(20))
    display(events.tail(30))


def add_signal_markers(fig: go.Figure, day_df: pd.DataFrame, row: int = 1, col: int = 1) -> None:
    if day_df.empty:
        return
    pad = max(float((day_df["high"] - day_df["low"]).median()) * 1.25, 10.0)
    calls = day_df.loc[day_df["signal"].eq("CALL")]
    puts = day_df.loc[day_df["signal"].eq("PUT")]
    if not calls.empty:
        fig.add_trace(
            go.Scatter(
                x=calls["timestamp"],
                y=calls["low"] - pad,
                mode="markers",
                marker=dict(symbol="triangle-up", color="#149447", size=11),
                name="CALL signal",
                hovertemplate="%{x}<br>CALL<extra></extra>",
            ),
            row=row,
            col=col,
        )
    if not puts.empty:
        fig.add_trace(
            go.Scatter(
                x=puts["timestamp"],
                y=puts["high"] + pad,
                mode="markers",
                marker=dict(symbol="triangle-down", color="#c73832", size=11),
                name="PUT signal",
                hovertemplate="%{x}<br>PUT<extra></extra>",
            ),
            row=row,
            col=col,
        )


def show_day_slice(strategy: pd.DataFrame, selected_day: str) -> pd.DataFrame:
    day_df = strategy.loc[strategy["session_date"].astype(str).eq(str(selected_day))].copy()
    if day_df.empty:
        available = ", ".join(strategy["session_date"].astype(str).unique()[:5])
        raise ValueError(f"No candles found for {selected_day}. First available days: {available}")
    print(f"{selected_day}: {len(day_df):,} candles, {day_df['signal'].ne('').sum():,} signal event(s)")
    return day_df


## Strategy Signals

In [ ]:
MEANREV_PARAMS = {
    "slope_window": 3,
    "vwap_band_multiplier": 1.0,
    "vwap_band_calc_mode": "standard_deviation",
    "supertrend_atr_period": 10,
    "supertrend_factor": 3.0,
    "rsi_length": 7,
    "rsi_ma_length": 14,
    "up_angle_ema_12": 60,
    "dn_angle_ema_12": -60,
    "up_angle_rsi_ma": 30,
    "dn_angle_rsi_ma": -30,
    "up_angle_vwap": 2,
    "dn_angle_vwap": -2,
    "vwap_band_lookback": 8,
}


def build_meanrev_vwap_supertrend(data: pd.DataFrame, params: dict) -> pd.DataFrame:
    pieces = []
    for _, day in data.groupby("session_date", sort=False):
        day = day.copy().reset_index(drop=True)
        day["bar_number"] = np.arange(1, len(day) + 1)
        vol = pd.to_numeric(day["effective_volume"], errors="coerce").fillna(0.0)
        day["fut_volume"] = vol
        day["hlc3"] = (day["high"] + day["low"] + day["close"]) / 3.0
        day["pv"] = day["hlc3"] * vol
        day["p2v"] = (day["hlc3"] ** 2) * vol
        day["cum_vol"] = vol.cumsum()
        day["cum_pv"] = day["pv"].cumsum()
        day["cum_p2v"] = day["p2v"].cumsum()
        valid_volume = day["cum_vol"] > 0
        day["vwap"] = np.nan
        day.loc[valid_volume, "vwap"] = day.loc[valid_volume, "cum_pv"] / day.loc[valid_volume, "cum_vol"]
        variance = pd.Series(np.nan, index=day.index, dtype="float64")
        variance.loc[valid_volume] = (
            day.loc[valid_volume, "cum_p2v"] / day.loc[valid_volume, "cum_vol"]
        ) - (day.loc[valid_volume, "vwap"] ** 2)
        stdev_abs = np.sqrt(variance.clip(lower=0.0))
        if params["vwap_band_calc_mode"] == "percentage":
            band_basis = day["vwap"].abs() * 0.01
        else:
            band_basis = stdev_abs
        day["upperbound"] = day["vwap"] + (band_basis * float(params["vwap_band_multiplier"]))
        day["lowerbound"] = day["vwap"] - (band_basis * float(params["vwap_band_multiplier"]))

        day = add_price_action(day, volatile_range=5.0, major_move_default=12.0)
        day["ema_12"] = ema(day["close"], 12)
        day["rsi_7"] = rsi(day["close"], int(params["rsi_length"]))
        day["rsi_ma_14"] = day["rsi_7"].rolling(window=int(params["rsi_ma_length"]), min_periods=int(params["rsi_ma_length"])).mean()
        day["angle_vwap"] = angle(day["vwap"], int(params["slope_window"]))
        day["angle_ema_12"] = angle(day["ema_12"], int(params["slope_window"]))
        day["angle_rsi_ma_14"] = angle(day["rsi_ma_14"], int(params["slope_window"]))

        st = calculate_supertrend(
            day["high"],
            day["low"],
            day["close"],
            atr_period=int(params["supertrend_atr_period"]),
            factor=float(params["supertrend_factor"]),
        )
        for col in ["st_atr_10", "st_upperbound", "st_lowerbound", "supertrend", "st_direction"]:
            day[col] = st[col].to_numpy()
        day["st_turn_green"] = (day["st_direction"] < 0) & (day["st_direction"].shift(1) > 0)
        day["st_turn_red"] = (day["st_direction"] > 0) & (day["st_direction"].shift(1) < 0)
        lookback = int(params["vwap_band_lookback"])
        below_lower_body = (day["open"] < day["lowerbound"]) & (day["close"] < day["lowerbound"])
        above_upper_body = (day["open"] > day["upperbound"]) & (day["close"] > day["upperbound"])
        day["last_candles_below_lower"] = (
            below_lower_body.shift(1).rolling(window=lookback, min_periods=lookback).max().fillna(False).astype(bool)
        )
        day["last_candles_above_upper"] = (
            above_upper_body.shift(1).rolling(window=lookback, min_periods=lookback).min().fillna(False).astype(bool)
        )
        after_start = day["timestamp"].dt.time >= time(9, 45)
        call_setup = (
            (day["bar_number"] >= 30)
            & after_start
            & day["st_turn_green"]
            & (day["st_direction"] < 0)
            & day["last_candles_below_lower"]
            & (day["angle_ema_12"] > params["up_angle_ema_12"])
            & (day["angle_rsi_ma_14"] > params["up_angle_rsi_ma"])
        )
        put_setup = (
            (day["bar_number"] >= 30)
            & after_start
            & day["st_turn_red"]
            & (day["st_direction"] > 0)
            & day["last_candles_above_upper"]
            & (day["angle_ema_12"] < params["dn_angle_ema_12"])
            & (day["angle_rsi_ma_14"] < params["dn_angle_rsi_ma"])
        )
        day["raw_signal"] = np.select([call_setup, put_setup], ["CALL", "PUT"], default="")
        day["signal"] = signal_events(day["raw_signal"])
        pieces.append(day)
    return pd.concat(pieces, ignore_index=True)


strategy = build_meanrev_vwap_supertrend(market_data, MEANREV_PARAMS)
display_signal_summary(
    strategy,
    [
        "timestamp",
        "signal",
        "close",
        "vwap",
        "upperbound",
        "lowerbound",
        "ema_12",
        "rsi_ma_14",
        "angle_ema_12",
        "angle_rsi_ma_14",
        "st_direction",
        "supertrend",
        "last_candles_below_lower",
        "last_candles_above_upper",
    ],
)


## Full Period Signal Overview

In [ ]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.72, 0.28],
    vertical_spacing=0.05,
    subplot_titles=("MeanRev VWAP Supertrend Signals", "VWAP / EMA / RSI MA Angles"),
)
fig.add_trace(go.Scatter(x=strategy["timestamp"], y=strategy["close"], name="Close", line=dict(color="#2b2b2b", width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=strategy["timestamp"], y=strategy["vwap"], name="VWAP", line=dict(color="#1f77b4", width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=strategy["timestamp"], y=strategy["upperbound"], name="VWAP Upper", line=dict(color="#7f8c8d", width=0.8, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=strategy["timestamp"], y=strategy["lowerbound"], name="VWAP Lower", line=dict(color="#7f8c8d", width=0.8, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=strategy["timestamp"], y=strategy["ema_12"], name="EMA 12", line=dict(color="#ff7f0e", width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=strategy["timestamp"], y=strategy["supertrend"], name="Supertrend", line=dict(color="#9467bd", width=1)), row=1, col=1)
add_signal_markers(fig, strategy, row=1, col=1)
fig.add_trace(go.Scatter(x=strategy["timestamp"], y=strategy["angle_vwap"], name="VWAP angle", line=dict(color="#1f77b4", width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=strategy["timestamp"], y=strategy["angle_ema_12"], name="EMA 12 angle", line=dict(color="#ff7f0e", width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=strategy["timestamp"], y=strategy["angle_rsi_ma_14"], name="RSI MA angle", line=dict(color="#238b45", width=1)), row=2, col=1)
fig.update_layout(height=720, template="plotly_white", hovermode="x unified", legend=dict(orientation="h", y=1.02))
fig.update_xaxes(rangeslider_visible=False)
fig.show()


## One Day Candlestick View

In [ ]:
ONE_DAY = first_signal_or_latest_day(strategy)
day_df = show_day_slice(strategy, ONE_DAY)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.62, 0.18, 0.20],
    vertical_spacing=0.04,
    subplot_titles=(f"{ONE_DAY} Candles With VWAP Bands And Supertrend", "Future Volume", "VWAP / EMA / RSI MA Angles"),
)
fig.add_trace(
    go.Candlestick(
        x=day_df["timestamp"],
        open=day_df["open"],
        high=day_df["high"],
        low=day_df["low"],
        close=day_df["close"],
        name="Nifty50",
        increasing_line_color="#149447",
        decreasing_line_color="#c73832",
    ),
    row=1,
    col=1,
)
fig.add_trace(go.Scatter(x=day_df["timestamp"], y=day_df["vwap"], name="VWAP", line=dict(color="#1f77b4", width=1.3)), row=1, col=1)
fig.add_trace(go.Scatter(x=day_df["timestamp"], y=day_df["upperbound"], name="VWAP Upper", line=dict(color="#7f8c8d", width=1, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=day_df["timestamp"], y=day_df["lowerbound"], name="VWAP Lower", line=dict(color="#7f8c8d", width=1, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=day_df["timestamp"], y=day_df["ema_12"], name="EMA 12", line=dict(color="#ff7f0e", width=1.2)), row=1, col=1)
fig.add_trace(go.Scatter(x=day_df["timestamp"], y=day_df["supertrend"], name="Supertrend", line=dict(color="#9467bd", width=1.2)), row=1, col=1)
add_signal_markers(fig, day_df, row=1, col=1)
fig.add_trace(go.Bar(x=day_df["timestamp"], y=day_df["fut_volume"], name="Future volume", marker_color="#7f8c8d"), row=2, col=1)
fig.add_trace(go.Scatter(x=day_df["timestamp"], y=day_df["angle_vwap"], name="VWAP angle", line=dict(color="#1f77b4", width=1.2)), row=3, col=1)
fig.add_trace(go.Scatter(x=day_df["timestamp"], y=day_df["angle_ema_12"], name="EMA 12 angle", line=dict(color="#ff7f0e", width=1.2)), row=3, col=1)
fig.add_trace(go.Scatter(x=day_df["timestamp"], y=day_df["angle_rsi_ma_14"], name="RSI MA angle", line=dict(color="#238b45", width=1.2)), row=3, col=1)
fig.update_layout(height=840, template="plotly_white", hovermode="x unified", legend=dict(orientation="h", y=1.02))
fig.update_xaxes(rangeslider_visible=False)
fig.show()
